<a href="https://colab.research.google.com/github/Toan-ten/Arxiv-Project-Group10/blob/feat%2Fm1-t02-colab-resume/notebooks/m1_colab_fast_production.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Quy trình thu thập dữ liệu chính thức giai đoạn 1 trên Google Colab

**Quy định vận hành:**
- **Môi trường:** Google Colab CPU-only (Runtime → Change runtime type → CPU / accelerator None). Tuyệt đối không dùng GPU/TPU.
- **Đơn tiến trình (Single-writer):** Mỗi corpus chỉ được ghi bởi một tiến trình duy nhất. Không chạy song song nhiều sổ tay cùng ghi vào một thư mục.
- **Bảo mật:** Khóa truy cập Kaggle (`KAGGLE_API_TOKEN`) và Semantic Scholar (`S2_API_KEY`) được nạp an toàn từ Google Colab Secrets (biểu tượng ổ khóa ở thanh bên trái), không viết trực tiếp vào mã nguồn.
- **Kiến trúc tăng tốc:**
  1. Thư mục làm việc cục bộ (`/content/fast_crawling_workspace`): Giảm thiểu độ trễ I/O của Google Drive FUSE.
  2. RAM Disk (`/dev/shm`): Giải nén và lọc tệp TeX/Bib trực tiếp trong bộ nhớ RAM.
  3. Nạp trước siêu dữ liệu Kaggle: Trích xuất lịch sử phiên bản và ngày nộp từ tệp snapshot cục bộ, tiết kiệm hàng nghìn truy vấn discovery.
  4. Sao lưu và khôi phục bền vững: Tự động nén tệp `.tar.gz` kèm mã băm `.sha256` đồng bộ lên Google Drive sau mỗi lô 100 bài báo.

In [2]:
# Bước 1: Thiết lập môi trường, kết nối Google Drive và kiểm tra mã nguồn
CODE_COMMIT = "de5e348fa6aada6ed3dd109006b26f9d88308e62"  # Điền mã commit 40 ký tự đã được nghiệm thu

import os
from pathlib import Path
import re
import subprocess
import sys

if os.environ.get("COLAB_GPU") == "1":
    raise RuntimeError("Chuyển accelerator về None/CPU trong Runtime trước khi chạy.")

from google.colab import drive
drive.mount('/content/drive')

REPO_DIR = Path('/content/Arxiv-Project-Group10')
if not REPO_DIR.exists():
    subprocess.run(["git", "clone", "https://github.com/Toan-ten/Arxiv-Project-Group10.git", str(REPO_DIR)], check=True)

os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

if re.fullmatch(r"[0-9a-f]{40}", CODE_COMMIT):
    subprocess.run(["git", "fetch", "origin"], check=True)
    subprocess.run(["git", "checkout", "--detach", CODE_COMMIT], check=True)
    current_head = subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip()
    print(f"Đã ghim chính xác commit: {current_head}")
else:
    print("Cảnh báo: Chưa điền full 40-character SHA, sử dụng HEAD hiện tại của nhánh.")

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
print("Hoàn tất chuẩn bị môi trường và thư viện phụ thuộc.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Đã ghim chính xác commit: de5e348fa6aada6ed3dd109006b26f9d88308e62
Hoàn tất chuẩn bị môi trường và thư viện phụ thuộc.


In [3]:
# Bước 2: Nạp cấu hình Kaggle API an toàn từ Colab Secrets và chuẩn bị snapshot
import os
from pathlib import Path

kaggle_token = os.environ.get("KAGGLE_API_TOKEN")
if not kaggle_token:
    try:
        from google.colab import userdata
        kaggle_token = userdata.get("KAGGLE_API_TOKEN")
    except Exception:
        pass

if kaggle_token:
    kaggle_dir = Path.home() / ".kaggle"
    kaggle_dir.mkdir(parents=True, exist_ok=True)
    (kaggle_dir / "access_token").write_text(kaggle_token.strip() + "\n")
    (kaggle_dir / "access_token").chmod(0o600)
    print("Đã cấu hình Kaggle access token an toàn từ Colab Secrets.")
else:
    print("Chưa cấu hình KAGGLE_API_TOKEN trong Colab Secrets. Bỏ qua tải tự động từ Kaggle.")

snapshot_file = Path("arxiv-metadata-oai-snapshot.json")
if not snapshot_file.exists() and kaggle_token:
    print("Đang tải bộ dữ liệu Cornell-University/arxiv từ Kaggle...")
    !kaggle datasets download -d Cornell-University/arxiv
    !unzip -q -o arxiv.zip
    print("Đã giải nén arxiv-metadata-oai-snapshot.json.")
elif snapshot_file.exists():
    print("Tệp arxiv-metadata-oai-snapshot.json đã sẵn sàng trên ổ đĩa.")

Đã cấu hình Kaggle access token an toàn từ Colab Secrets.
Tệp arxiv-metadata-oai-snapshot.json đã sẵn sàng trên ổ đĩa.


In [4]:
# Bước 3: Thực thi thu thập mã nguồn TeX và lịch sử phiên bản (Lô 100 định danh)
# Phân bổ 200 lô dữ liệu:
# - Lô 1..50:    24120010 (Trần Văn Tấn Khôi)
# - Lô 51..100:  24120035 (Lê Thành Đạt)
# - Lô 101..150: 24120232 (Nguyễn Văn Toàn)
# - Lô 151..200: 24120260 (Trương Tuấn Anh)
import sys, os
from pathlib import Path
REPO_DIR = Path('/content/Arxiv-Project-Group10')
if REPO_DIR.exists():
    os.chdir(REPO_DIR)
    if str(REPO_DIR) not in sys.path:
        sys.path.insert(0, str(REPO_DIR))

from src.fast_production import FastProductionRunner

LOCAL_ROOT = '/content/fast_crawling_workspace'
DRIVE_ROOT = '/content/drive/MyDrive/Group10_M1_Production'

runner = FastProductionRunner(
    local_root=LOCAL_ROOT,
    drive_root=DRIVE_ROOT,
    retry_429=True,
    retry_wait_budget=300.0,
    timeout=30.0,
)

COHORT_LIST = [151]  # Điều chỉnh danh sách lô cần thực thi

for cohort_num in COHORT_LIST:
    restored = runner.restore_cohort(cohort_num)
    if restored > 0:
        print(f'Đã khôi phục {restored} thư mục bài báo của lô {cohort_num} từ bản sao lưu Google Drive.')

snapshot_path = Path('arxiv-metadata-oai-snapshot.json')
if snapshot_path.exists():
    print(f'Đang nạp trước siêu dữ liệu từ snapshot cho các lô {COHORT_LIST}...')
    res = runner.preload_kaggle_discovery(snapshot_path, COHORT_LIST)
    if not res.get('interrupted'):
        print(f'Nạp thành công {res["found_papers"]}/{res["target_papers"]} bài báo trong {res.get("elapsed_seconds", 0.0):.2f}s.')

for cohort_num in COHORT_LIST:
    print(f'\nBắt đầu thu thập lô {cohort_num} (100 định danh, revision_history=True)...')
    report = runner.run_cohort(cohort_num, resume=True, revision_history=True)
    stats = report.get('fast_runner', {})
    completed = stats.get('completed_papers', 0)
    elapsed = stats.get('elapsed_seconds', 0.0)
    print(f'Lô {cohort_num}: hoàn thành {completed}/100 bài báo (thời gian phiên: {elapsed:.1f}s).')
    print(f'Bản sao lưu trên Drive: {stats.get("archive_path")}')
    if stats.get('stopped'):
        print(f'Dừng chuỗi xử lý do điều kiện bảo vệ: {stats.get("stop_reason")}')
        break

Đang nạp trước siêu dữ liệu từ snapshot cho các lô [151]...
Nạp thành công 100/100 bài báo trong 19.61s.

Bắt đầu thu thập lô 151 (100 định danh, revision_history=True)...
Lô 151: hoàn thành 99/100 bài báo (thời gian phiên: 602.7s).
Bản sao lưu trên Drive: /content/drive/MyDrive/Group10_M1_Production/archives/cohort_0151_corpus.tar.gz


In [5]:
# Bước 4: Làm giàu dữ liệu trích dẫn và siêu dữ liệu Semantic Scholar (S2 Enrichment)
import os

s2_api_key = os.environ.get("S2_API_KEY")
if not s2_api_key:
    try:
        from google.colab import userdata
        s2_api_key = userdata.get("S2_API_KEY")
    except Exception:
        pass

if s2_api_key:
    os.environ["S2_API_KEY"] = s2_api_key.strip()
    print("Đã kích hoạt chế độ có xác thực Semantic Scholar API.")
    s2_pacing = 1.0
else:
    print("Chạy ở chế độ không xác thực Semantic Scholar (giới hạn 100 requests / 5 phút). Pacing: 3.0s.")
    s2_pacing = 3.0

s2_cfg = {
    "timeout": 30.0,
    "max_attempts": 3,
    "min_interval": s2_pacing,
    "reference_limit": 100,
}

for cohort_num in COHORT_LIST:
    print(f'\nBắt đầu làm giàu dữ liệu Semantic Scholar cho lô {cohort_num}...')
    s2_report = runner.enrich_cohort_s2(cohort_num, s2_config=s2_cfg)
    stats = s2_report.get('fast_runner', {})
    print(f'Lô {cohort_num}: đã đồng bộ dữ liệu S2 hoàn tất lên Drive tại {stats.get("archive_path")}.')

Đã kích hoạt chế độ có xác thực Semantic Scholar API.

Bắt đầu làm giàu dữ liệu Semantic Scholar cho lô 151...
Lô 151: Đã có 99/100 bài báo hoàn thành từ phiên trước, tiếp tục thu thập các bài còn lại.
Lô 151: đã đồng bộ dữ liệu S2 hoàn tất lên Drive tại /content/drive/MyDrive/Group10_M1_Production/archives/cohort_0151_corpus.tar.gz.


In [6]:
# Bước 5: Kiểm toán chất lượng dữ liệu và tạo bảng báo cáo Milestone 1
import sys, os
from pathlib import Path
REPO_DIR = Path('/content/Arxiv-Project-Group10')
if REPO_DIR.exists():
    os.chdir(REPO_DIR)
    if str(REPO_DIR) not in sys.path:
        sys.path.insert(0, str(REPO_DIR))

from scripts.generate_m1_report import compute_m1_statistics, format_markdown_summary
from src.fast_production import cohort_pairs

all_pairs = []
for c in COHORT_LIST:
    all_pairs.extend(cohort_pairs(c))

stats = compute_m1_statistics(runner.store, all_pairs)
report_md = format_markdown_summary(stats)

report_out = Path(DRIVE_ROOT) / "reports" / f"m1_audit_summary_cohort_{COHORT_LIST[0]:04d}.md"
report_out.parent.mkdir(parents=True, exist_ok=True)
report_out.write_text(report_md, encoding="utf-8")
print(f"Đã xuất bảng báo cáo kiểm toán ra Google Drive: {report_out}\n")
print(report_md)

Đã xuất bảng báo cáo kiểm toán ra Google Drive: /content/drive/MyDrive/Group10_M1_Production/reports/m1_audit_summary_cohort_0151.md

### Báo cáo thống kê thu thập dữ liệu Milestone 1

#### 1. Xuất xứ kiểm toán (Audit Provenance)

| Thuộc tính | Giá trị |
|---|---|
| Thời điểm tạo báo cáo | `2026-11-09T06:41:39.801478+00:00` |
| Mã băm mã nguồn (Commit) | `UNKNOWN` |
| Thư mục lưu trữ Corpus | `/content/fast_crawling_workspace/corpus` |
| Phiên bản định nghĩa số liệu | `v2.0.0` |

#### 2. Chỉ số thu thập và chất lượng dữ liệu (Canonical Metrics)

| Chỉ số kiểm toán M1 | Số lượng | Tỷ lệ (%) / Ghi chú |
|---|---:|---:|
| Tổng số định danh phân công | 100 | 100.0% |
| Bài báo tồn tại trên arXiv (found) | 100 | Ứng viên thực tế |
| Bài báo không tồn tại (not_found) | 0 | Định danh trống |
| Bài báo hoàn tất mã nguồn (source_complete) | 99 | 99.0% bài tìm thấy |
| Bài báo hoàn tất toàn bộ Milestone 1 (m1_complete) | 0 | 0.0% bài tìm thấy |
| Bài báo thu thập dở dang (partial) | 100 | Cần r

In [7]:
# Bước 6: Xuất tập dữ liệu chính thức (papers & references) dạng Parquet và CSV
import sys, os
from pathlib import Path
REPO_DIR = Path('/content/Arxiv-Project-Group10')
if REPO_DIR.exists():
    os.chdir(REPO_DIR)
    if str(REPO_DIR) not in sys.path:
        sys.path.insert(0, str(REPO_DIR))

DRIVE_ROOT = '/content/drive/MyDrive/Group10_M1_Production'
from src.export import export_papers_table, export_references_table, export_to_parquet, validate_exported_tables
import pandas as pd

export_dir = Path(DRIVE_ROOT) / "final_output"
export_dir.mkdir(parents=True, exist_ok=True)

papers_rows = export_papers_table(runner.store, all_pairs)
refs_rows = export_references_table(runner.store, all_pairs)

val_res = validate_exported_tables(papers_rows, refs_rows)
print("Kết quả kiểm tra hợp chuẩn schema xuất bản:", val_res)
if not val_res["is_valid"]:
    print("Cảnh báo vi phạm schema:", val_res["issues"])

export_to_parquet(papers_rows, export_dir / "papers.parquet")
export_to_parquet(refs_rows, export_dir / "references.parquet")
pd.DataFrame(papers_rows).to_csv(export_dir / "papers.csv", index=False)
pd.DataFrame(refs_rows).to_csv(export_dir / "references.csv", index=False)
print(f"Đã xuất thành công {len(papers_rows)} bài báo và {len(refs_rows)} quan hệ trích dẫn ra thư mục: {export_dir}")

Kết quả kiểm tra hợp chuẩn schema xuất bản: {'is_valid': True, 'total_papers': 99, 'total_references': 0, 'unique_source_papers_with_refs': 0, 'issue_count': 0, 'issues': []}
Đã xuất thành công 99 bài báo và 0 quan hệ trích dẫn ra thư mục: /content/drive/MyDrive/Group10_M1_Production/final_output


In [8]:
import shutil
from pathlib import Path

# 1. Xóa sạch dữ liệu cục bộ trên Colab
local_workspace = Path("/content/fast_crawling_workspace")
if local_workspace.exists():
    shutil.rmtree(local_workspace)
    print("Đã xóa sạch thư mục cục bộ:", local_workspace)

scratch_shm = Path("/dev/shm/arxiv_scratch")
if scratch_shm.exists():
    shutil.rmtree(scratch_shm)
    print("Đã dọn dẹp vùng nhớ tạm RAM disk:", scratch_shm)

# 2. Xóa sạch bản sao lưu cũ trên Google Drive
drive_production = Path("/content/drive/MyDrive/Group10_M1_Production")
if drive_production.exists():
    shutil.rmtree(drive_production)
    print("Đã xóa sạch toàn bộ dữ liệu trên Google Drive:", drive_production)

print("\nHoàn tất dọn dẹp! Toàn bộ dữ liệu cũ đã bị xóa sạch.")


Đã xóa sạch thư mục cục bộ: /content/fast_crawling_workspace
Đã dọn dẹp vùng nhớ tạm RAM disk: /dev/shm/arxiv_scratch
Đã xóa sạch toàn bộ dữ liệu trên Google Drive: /content/drive/MyDrive/Group10_M1_Production

Hoàn tất dọn dẹp! Toàn bộ dữ liệu cũ đã bị xóa sạch.
